In [1]:
import requests
import time

base_url = "https://collectionapi.metmuseum.org"
search_path = "/public/collection/v1.1/search"

search_query = "Sioux"
limit = 500
offset = 0

all_results = []

while True:
    url = (
        f"{base_url}{search_path}"
        f"?q={search_query.replace(' ', '%20')}"
        f"&artistOrCulture=true"
        f"&limit={limit}"
        f"&offset={offset}"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    # Get the object IDs returned by this page
    object_ids = data.get("objectIDs") or []

    # Stop if there are no more results
    if not object_ids:
        break

    all_results.extend(object_ids)

    print(f"Collected {len(all_results)} records so far")

    # Move to the next page
    offset += limit

    time.sleep(1)

print(f"Total records collected: {len(all_results)}")

Collected 48 records so far
Total records collected: 48


In [2]:
print(url)

https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=Sioux&artistOrCulture=true&limit=500&offset=500


In [3]:
complete_sioux_objects = all_results

print(len(complete_sioux_objects))


48


In [5]:
from time import sleep 

object_path = "/public/collection/v1/objects/" 
all_sioux_details = []

for sioux_artwork in complete_sioux_objects:
    url = f"{base_url}{object_path}{sioux_artwork}"
    response_object = requests.get(url) 
    
    # Check if the API request was successful (status code 200)
    if response_object.ok:
        all_sioux_details.append(response_object.json())
        print(f"grabbed details from objectID {sioux_artwork} with status code {response_object.status_code}")
    else:
        # If it's a 404 or any other error, skip it and don't append it to the list
        print(f"⚠️ SKIPPED objectID {sioux_artwork} - Received status code {response_object.status_code}")
        
    sleep(1) 

print(f"Total valid objects captured: {len(all_sioux_details)}") 
if all_sioux_details:
    print(all_sioux_details[0])

⚠️ SKIPPED objectID 822647 - Received status code 404
grabbed details from objectID 501247 with status code 200
grabbed details from objectID 501250 with status code 200
grabbed details from objectID 313250 with status code 200
grabbed details from objectID 313288 with status code 200
grabbed details from objectID 318349 with status code 200
grabbed details from objectID 318351 with status code 200
grabbed details from objectID 319139 with status code 200
grabbed details from objectID 319146 with status code 200
grabbed details from objectID 319148 with status code 200
grabbed details from objectID 827407 with status code 200
grabbed details from objectID 318347 with status code 200
grabbed details from objectID 319027 with status code 200
grabbed details from objectID 319037 with status code 200
grabbed details from objectID 319143 with status code 200
grabbed details from objectID 319188 with status code 200
grabbed details from objectID 319029 with status code 200
grabbed details fr

In [6]:
import csv   # brings in all the csv-related functions

with open("all_sioux_artworks.csv", "w", encoding="utf-8") as file: # initiate the opening of a file, and write a new file
    writer = csv.DictWriter(file, all_sioux_details[0].keys()) # this creates the columns in the CSV file from the keys
    writer.writeheader() # Creates the first row of the CSV, containing the HEaders
    writer.writerows(all_sioux_details) # goes through every item in the yoko_ono_artworks list and put it in the CSV file
    # you should see it in your folder after running.

In [7]:
sioux_with_state = [
    artwork for artwork in all_sioux_details
    if artwork.get("state")
]

In [8]:
print(f"Original records: {len(all_sioux_details)}")
print(f"Records with state: {len(sioux_with_state)}")

Original records: 47
Records with state: 25


In [10]:
import csv   # brings in all the csv-related functions

with open("sioux_artworks_withstate.csv", "w", encoding="utf-8") as file: # initiate the opening of a file, and write a new file
    writer = csv.DictWriter(file, sioux_with_state[0].keys()) # this creates the columns in the CSV file from the keys
    writer.writeheader() # Creates the first row of the CSV, containing the HEaders
    writer.writerows(sioux_with_state) # goes through every item in the yoko_ono_artworks list and put it in the CSV file
    # you should see it in your folder after running.

In [9]:
sioux_with_state_filtered = [
    artwork for artwork in all_sioux_details
    if artwork.get("state")
    and not any(term in artwork["state"] for term in ["and", "or", "/", ",", "?", "&"])
]

In [11]:
print(f"Original records: {len(all_sioux_details)}")
print(f"Records with state: {len(sioux_with_state)}")
print(f"Records with state: {len(sioux_with_state_filtered)}")

Original records: 47
Records with state: 25
Records with state: 11


In [12]:
import csv   # brings in all the csv-related functions

with open("sioux_artworks_withstate_filtered.csv", "w", encoding="utf-8") as file: # initiate the opening of a file, and write a new file
    writer = csv.DictWriter(file, sioux_with_state[0].keys()) # this creates the columns in the CSV file from the keys
    writer.writeheader() # Creates the first row of the CSV, containing the HEaders
    writer.writerows(sioux_with_state_filtered) # goes through every item in the yoko_ono_artworks list and put it in the CSV file
    # you should see it in your folder after running.

In [13]:
# Pulling the coordinates:


import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Read your CSV
df = pd.read_csv("sioux_artworks_withstate_filtered.csv")

# Clean up state names
df["state"] = df["state"].str.strip()

# Initialize OpenStreetMap geocoder
geolocator = Nominatim(user_agent="sioux_artwork_geocoder")

# Add a 1-second delay between requests
geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1
)

# Dictionary to store coordinates
coordinates = {}

# Geocode each unique state
for state in df["state"].dropna().unique():

    # Skip regional labels that aren't states
    if state in ["Great Plains", "Dakotas"]:
        continue

    location = geocode(f"{state}, USA")

    if location:
        coordinates[state] = {
            "latitude": location.latitude,
            "longitude": location.longitude
        }
    else:
        coordinates[state] = {
            "latitude": None,
            "longitude": None
        }

# Add latitude and longitude columns
df["latitude"] = df["state"].map(
    lambda x: coordinates.get(x, {}).get("latitude")
)

df["longitude"] = df["state"].map(
    lambda x: coordinates.get(x, {}).get("longitude")
)

# Export updated CSV
df.to_csv("sioux_artworks_with_coordinates.csv", index=False)

print(df[["state", "latitude", "longitude"]])


ModuleNotFoundError: No module named 'geopy'